# counseLLM — Evaluation & Results

Compare all three model variants:
1. **Base** — Llama 3.1 8B Instruct (no fine-tuning)
2. **SFT** — After supervised fine-tuning on counseling data
3. **SFT+DPO** — After preference alignment

Evaluation includes automated metrics and LLM-as-judge scoring.

In [ ]:
!pip install torch transformers peft bitsandbytes matplotlib seaborn -q

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

## 1. Run Evaluation

Generate responses and compute metrics using the evaluation scripts.

In [ ]:
# Run automated evaluation (generates responses + metrics for all 3 variants)
!python ../eval/evaluate.py --models base sft dpo

## 2. Automated Metrics

In [ ]:
# Load evaluation results
results_file = RESULTS_DIR / "eval_results.json"
with open(results_file) as f:
    results = json.load(f)

variants = list(results.keys())
print(f"Variants: {variants}")

# Metrics comparison table
metrics_data = {}
for v in variants:
    m = results[v]["metrics"]
    metrics_data[v] = {
        "Avg Words": m["length_stats"]["avg_words"],
        "Distinct-1": m["distinct_1"],
        "Distinct-2": m["distinct_2"],
        "Perplexity": m["perplexity"],
    }

# Print table
print(f"\n{'Metric':<20}", end="")
for v in variants:
    print(f"{v:>12}", end="")
print()
print("-" * (20 + 12 * len(variants)))
for metric in ["Avg Words", "Distinct-1", "Distinct-2", "Perplexity"]:
    print(f"{metric:<20}", end="")
    for v in variants:
        print(f"{metrics_data[v][metric]:>12.2f}", end="")
    print()

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#4A90D9", "#E8833A", "#50B88C"]

# Avg response length
axes[0].bar(variants, [metrics_data[v]["Avg Words"] for v in variants], color=colors)
axes[0].set_title("Average Response Length (words)")
axes[0].set_ylabel("Words")

# Distinct-2 (diversity)
axes[1].bar(variants, [metrics_data[v]["Distinct-2"] for v in variants], color=colors)
axes[1].set_title("Response Diversity (Distinct-2)")
axes[1].set_ylabel("Ratio")

# Perplexity
axes[2].bar(variants, [metrics_data[v]["Perplexity"] for v in variants], color=colors)
axes[2].set_title("Perplexity (lower is better)")
axes[2].set_ylabel("Perplexity")

plt.tight_layout()
plt.show()

## 3. LLM-as-Judge Results

Run the LLM judge first (requires API key):
```bash
python eval/llm_judge.py --from-results results/eval_results.json --provider anthropic --api-key $ANTHROPIC_API_KEY
```

In [ ]:
# Load LLM judge results
judge_file = RESULTS_DIR / "judge_results.json"
if judge_file.exists():
    with open(judge_file) as f:
        judge_results = json.load(f)

    summary = judge_results["summary"]
    dims = ["empathy", "safety", "relevance", "helpfulness"]
    variants_j = list(summary.keys())

    # Radar chart
    angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors_r = ["#4A90D9", "#E8833A", "#50B88C"]

    for i, v in enumerate(variants_j):
        values = [summary[v][d] for d in dims]
        values += values[:1]
        ax.plot(angles, values, "o-", linewidth=2, label=v, color=colors_r[i])
        ax.fill(angles, values, alpha=0.1, color=colors_r[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([d.capitalize() for d in dims], size=12)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    ax.set_title("LLM Judge Scores by Dimension", size=14, pad=20)
    plt.tight_layout()
    plt.show()

    # Print summary table
    print(f"\n{'Dimension':<20}", end="")
    for v in variants_j:
        print(f"{v:>12}", end="")
    print()
    print("-" * (20 + 12 * len(variants_j)))
    for d in dims:
        print(f"{d:<20}", end="")
        for v in variants_j:
            print(f"{summary[v][d]:>12.2f}", end="")
        print()
    print("-" * (20 + 12 * len(variants_j)))
    print(f"{'OVERALL':<20}", end="")
    for v in variants_j:
        avg = sum(summary[v][d] for d in dims) / len(dims)
        print(f"{avg:>12.2f}", end="")
    print()
else:
    print("Run llm_judge.py first to generate judge results.")

## 4. Per-Category Analysis

In [ ]:
if judge_file.exists():
    detailed = judge_results["detailed_scores"]
    dims = ["empathy", "safety", "relevance", "helpfulness"]

    # Per-category heatmap for each variant
    categories = sorted(set(s["category"] for s in detailed[variants_j[0]] if not s.get("error")))

    for v in variants_j:
        cat_dim_scores = {}
        for cat in categories:
            cat_scores = [s for s in detailed[v] if s["category"] == cat and not s.get("error")]
            if cat_scores:
                cat_dim_scores[cat] = {d: sum(s[d] for s in cat_scores) / len(cat_scores) for d in dims}

        data = [[cat_dim_scores.get(cat, {}).get(d, 0) for d in dims] for cat in categories]

        plt.figure(figsize=(8, max(6, len(categories) * 0.4)))
        sns.heatmap(
            data, annot=True, fmt=".1f", cmap="RdYlGn", vmin=1, vmax=5,
            xticklabels=[d.capitalize() for d in dims],
            yticklabels=categories,
        )
        plt.title(f"Judge Scores by Category — {v}")
        plt.tight_layout()
        plt.show()
else:
    print("Run llm_judge.py first.")

## 5. Side-by-Side Response Comparison

View actual responses from all three variants for selected prompts.

In [ ]:
# Show side-by-side comparisons for key categories
highlight_ids = ["anxiety_01", "depression_01", "crisis_01", "grief_01", "relationship_01"]

comparisons_dir = RESULTS_DIR / "comparisons"
if comparisons_dir.exists():
    for pid in highlight_ids:
        comp_file = comparisons_dir / f"{pid}.json"
        if not comp_file.exists():
            continue
        
        with open(comp_file) as f:
            comp = json.load(f)
        
        print(f"\n{'='*80}")
        print(f"[{comp['category'].upper()}] {comp['id']}")
        print(f"{'='*80}")
        print(f"\nPROMPT: {comp['prompt'][:200]}...")
        
        for variant, response in comp["responses"].items():
            print(f"\n--- {variant.upper()} ---")
            print(response[:500])
            if len(response) > 500:
                print("...")
        print()
else:
    print("Run evaluate.py first to generate comparisons.")

## 6. Training Loss Curves

If you used W&B, check your dashboard. Otherwise, load from trainer state.

In [ ]:
# Plot training loss from trainer_state.json (if available)
for stage, stage_dir in [("SFT", "outputs/sft"), ("DPO", "outputs/dpo")]:
    state_file = PROJECT_ROOT / stage_dir / "trainer_state.json"
    if not state_file.exists():
        # Check subdirectories for checkpoints
        for p in (PROJECT_ROOT / stage_dir).glob("*/trainer_state.json"):
            state_file = p
            break
    
    if state_file.exists():
        with open(state_file) as f:
            state = json.load(f)
        
        log_history = state.get("log_history", [])
        train_steps = [e["step"] for e in log_history if "loss" in e]
        train_loss = [e["loss"] for e in log_history if "loss" in e]
        eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
        eval_loss = [e["eval_loss"] for e in log_history if "eval_loss" in e]
        
        plt.figure(figsize=(10, 5))
        if train_steps:
            plt.plot(train_steps, train_loss, label="Train Loss", alpha=0.7)
        if eval_steps:
            plt.plot(eval_steps, eval_loss, label="Eval Loss", marker="o", markersize=4)
        plt.title(f"{stage} — Training Loss Curve")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f"{stage}: No trainer_state.json found. Run training first or check W&B dashboard.")